# Hugging Face Veri Seti Kalite Analizi

## tl;dr

- **45 veri setinde 87,831 satır** yerel sabit
  kopyalardan profillendi.
- Şema dağılımı: **catalog: 1, conversation: 43, tabular: 1**. Sohbet veya düz istem–cevap
  biçimindeki toplam kayıt sayısı **87,228**.
- Ayrı `thinking` içeren kayıt sayısı **67,470**; metin
  biçiminde saklanan `null` alan sayısı **462**.
- Erişim engeli nedeniyle analiz edilemeyen veri seti sayısı
  **2**; bunlar sessizce atlanmadı, kanıtlarıyla
  birlikte ayrı bir tabloda listeleniyor.
- Tekrar, eksik değer, metin uzunluğu ve zaman hassasiyeti ölçümleri
  kullanım amacına göre yorumlanmalı; satır hacmi tek başına kalite
  göstergesi değildir.

## Bağlam ve Yöntem

Analiz, Hugging Face Dataset Viewer üzerinden indirilen tam yerel
kopyalar ve sabit kaynak revizyonları üzerinde çalışır. Kontroller;
şema, tamamlılık, rol sırası, birebir ve normalleştirilmiş tekrar,
metin uzunluğu, `thinking`, metin biçimindeki `null`, kişisel veri
biçimleri ve zaman hassasiyeti sinyallerini kapsar.

### Temel Varsayımlar

- Normalleştirilmiş tekrar oranı, her metin ailesindeki ilk örnekten
  sonraki ek kopyaların tüm satırlara bölünmesiyle hesaplanır.
- Zaman hassasiyeti sayıları regex eşleşmeleridir; benzersiz satır
  sayısı değildir.
- Regex taraması bağlamsal kişi ve kurum adlarını bütünüyle yakalayamaz.
- Yapısal doğrulama, her cevabın alan uzmanı tarafından olgusal olarak
  onaylandığı anlamına gelmez.

## Veri

In [1]:
import json
from pathlib import Path
import pandas as pd

WORKING_DIR = Path.cwd()
PROJECT_DIR = next(
    (
        candidate
        for candidate in (WORKING_DIR, WORKING_DIR.parent)
        if (candidate / "outputs" / "data_quality_profiles.json").exists()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project outputs were not found in the working directory or its parent."
    )

profiles = json.loads(
    (PROJECT_DIR / "outputs" / "data_quality_profiles.json").read_text(encoding="utf-8")
)
overlaps = json.loads(
    (PROJECT_DIR / "outputs" / "cross_dataset_overlap.json").read_text(encoding="utf-8")
)
manual_path = PROJECT_DIR / "outputs" / "manual_findings.json"
manual = json.loads(manual_path.read_text(encoding="utf-8")) if manual_path.exists() else {"datasets": {}}
excluded = json.loads(
    (PROJECT_DIR / "outputs" / "excluded_datasets.json").read_text(encoding="utf-8")
)["datasets"]

len(profiles), sum(item["profile"]["row_count"] for item in profiles)

(45, 87831)

### Analiz edilemeyen veri setleri

Kayıt defterinde etkin olduğu hâlde erişilemeyen veri setleri kapsam
dışı bırakılmaz; her biri canlı olarak yeniden doğrulanmış HTTP
kanıtıyla birlikte burada listelenir. Aşağıdaki satırlar toplam satır
sayısına dâhil değildir.

In [2]:
pd.DataFrame([
    {
        "dataset": item["dataset_id"],
        "contributor": item.get("contributor"),
        "status": item["status"],
        "reverified_on": item.get("reverified_on"),
        "http_status": ", ".join(
            str(check.get("status")) for check in item.get("reverified_checks", [])
        ),
    }
    for item in excluded
])

,dataset,contributor,status,reverified_on,http_status
0,menesnas/Pharmacy_Identity_Synthetic_QA,Muhammet Enes Nas,gated_manual,2026-07-22,"200, 404"
1,uzcaliskan/magibu_dataset_drilling,Oguz Caliskan,not_found_or_private,2026-07-22,"404, 404"


In [3]:
summary_rows = []
for item in profiles:
    profile = item["profile"]
    scan = profile.get("text_scan", {})
    summary_rows.append({
        "dataset": item["dataset_id"],
        "contributor": item.get("contributor"),
        "rows": profile["row_count"],
        "shape": profile["data_shape"],
        "exact_duplicates": profile.get("exact_duplicate_rows", 0),
        "prompt_duplicates_%": round(100 * profile.get("user_prompt_duplicates", {}).get("duplicate_rate", 0), 2),
        "answer_duplicates_%": round(100 * profile.get("assistant_answer_duplicates", {}).get("duplicate_rate", 0), 2),
        "thinking": profile.get("nonempty_thinking_messages", 0),
        "string_nulls": profile.get("string_encoded_null_fields", 0),
        "time_sensitive_matches": scan.get("time_sensitive_matches", 0),
        "answer_words_median": profile.get("assistant_word_length", {}).get("median"),
        "answer_words_p95": profile.get("assistant_word_length", {}).get("p95"),
        "near_dup_answers_%": round(100 * (profile.get("assistant_answer_near_duplicates", {}).get("near_duplicate_rate") or 0), 2),
        "distinct_answers": profile.get("assistant_answer_duplicates", {}).get("distinct_values"),
        "distinct_rows": profile.get("distinct_canonical_rows"),
        "language_signal": profile.get("text_scan", {}).get("dominant_language_signal"),
        "topic_concentration_%": round(100 * ((profile.get("topic_profile") or {}).get("topic_concentration") or 0), 1),
        "distinctive_terms": ", ".join(
            t["term"] for t in (profile.get("topic_profile") or {}).get("distinctive_terms", [])[:5]
        ),
        "json_answers": profile.get("structured_answers", {}).get("json_parsable_answers", 0),
        "json_like_broken": profile.get("structured_answers", {}).get("json_like_but_unparsable_answers", 0),
        "conflicting_prompt_families": profile.get("answer_families", {}).get(
            "prompt_families_with_conflicting_answers", 0
        ),
        "cross_split_duplicate_rows": profile.get("split_integrity", {}).get(
            "cross_split_duplicate_rows", 0
        ),
    })

summary = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary

,dataset,contributor,rows,shape,exact_duplicates,prompt_duplicates_%,answer_duplicates_%,thinking,string_nulls,time_sensitive_matches,...,near_dup_answers_%,distinct_answers,distinct_rows,language_signal,topic_concentration_%,distinctive_terms,json_answers,json_like_broken,conflicting_prompt_families,cross_split_duplicate_rows
0,Aysenur44/namaz-vakti-dua-asistan-tr,Ayşe Nur Yeşilova,60,conversation,0,0.00,0.00,0,0,0,...,6.67,60.0,60,turkish,58.9,"namazı, hocam, detaylı, kılınabilir, vakti",0,0,0,0
1,Aysenur44/namaz-vakti-identity-tr,Ayşe Nur Yeşilova,4,conversation,0,0.00,0.00,0,0,0,...,0.00,4.0,4,turkish,100.0,"yararsın, adın, kimsin, seni, geliştirdi",0,0,0,0
2,Egertekin/marvel-domain-dataset,Ege Ertekin,177,conversation,0,75.71,13.56,0,0,1,...,13.56,153.0,177,turkish,80.6,"örümcek, spider, adam, detaylı, marvel",0,0,7,0
3,Endezyar/siyer_datasets,Muhammed Bakır Kurt,509,conversation,0,0.98,0.39,0,0,0,...,0.39,507.0,509,turkish,24.1,"efendimiz, müslüman, peygamberimiz, kabile, hi...",0,0,3,0
4,Erenyanic/seasoned-advice-dataset,Eren Yanic,1000,conversation,0,0.00,0.00,1000,0,28,...,0.00,1000.0,1000,turkish,7.0,"cooking, water, pişirme, some, recipe",0,24,0,0
5,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,Mert Ali Alkan,186,conversation,0,0.00,0.00,186,0,31,...,0.00,186.0,186,turkish,15.3,"sipariş, kargo, fatura, iade, iptal",0,0,0,0
6,SalihHub/trendyol-marangoz-urun-asistan-qa,Salih Dede,1211,conversation,0,1.32,15.11,1211,0,10,...,9.41,1028.0,1211,turkish,22.5,"merhaba, acaba, ürün, ürünün, duvara",0,0,12,0
7,Talayhan/skatepal_dataset,Talayhan,299,conversation,0,0.00,0.00,299,0,26,...,0.00,299.0,299,turkish,11.9,"kaykay, skateboard, skate, board, kickflip",0,0,0,0
8,Toivo0/Turkce-istatistik-reasoning,Umut Kıvanç Sipahioglu,400,conversation,0,0.00,0.00,400,0,12,...,0.00,400.0,400,turkish,16.3,"dağılımı, istatistiksel, olasılık, değişken, ö...",0,0,0,0
9,Uunan/turkish-cuisine-qa,Uunan,34244,conversation,0,0.01,0.05,34244,0,0,...,1.46,34227.0,34244,turkish,31.8,"tarifinde, aittir, çorbası, yapım, öğün",0,0,2,0


## Sonuçlar

### Portföy toplamları yeniden hesaplanıyor

Bu hücre temel toplamları doğrudan profil dosyasından üretir ve
notebook oluşturulurken gözlenen sabit değerlerle karşılaştırır.

In [4]:
portfolio = pd.Series({
    "dataset_count": len(summary),
    "total_rows": int(summary["rows"].sum()),
    "conversation_rows": int(summary.loc[summary["shape"] == "conversation", "rows"].sum()),
    "instruction_pair_rows": int(summary.loc[summary["shape"] == "instruction_pair", "rows"].sum()),
    "catalog_rows": int(summary.loc[summary["shape"] == "catalog", "rows"].sum()),
    "tabular_rows": int(summary.loc[summary["shape"] == "tabular", "rows"].sum()),
    "thinking_records": int(summary["thinking"].sum()),
    "string_null_fields": int(summary["string_nulls"].sum()),
    "time_sensitive_matches": int(summary["time_sensitive_matches"].sum()),
}, name="value")
assert portfolio["dataset_count"] == 45
assert portfolio["total_rows"] == 87831
portfolio.to_frame()

,value
dataset_count,45
total_rows,87831
conversation_rows,87228
instruction_pair_rows,0
catalog_rows,103
tabular_rows,500
thinking_records,67470
string_null_fields,462
time_sensitive_matches,3853


### Tekrar yoğunluğu veri setine göre değişiyor

Aşağıdaki tablo üç ölçümü birlikte gösterir. İlk ikisi yalnız büyük
harf ve noktalama farklarını yok sayan tam eşleşmedir.
`near_dup_answers_%` ise bir cevabın daha önceki bir cevapla en az
%85 token örtüşmesi taşıdığı satırları sayar; yeniden yazılmış
kopyaları yakalar. `distinct_answers` sütunu veri setinin gerçekte
kaç farklı cevap içerdiğini verir.

In [5]:
summary.loc[
    summary["shape"].isin(["conversation", "instruction_pair"]),
    ["dataset", "rows", "distinct_answers", "prompt_duplicates_%",
     "answer_duplicates_%", "near_dup_answers_%"],
].sort_values(
    ["answer_duplicates_%", "near_dup_answers_%"], ascending=False
)

,dataset,rows,distinct_answers,prompt_duplicates_%,answer_duplicates_%,near_dup_answers_%
19,enes1863/bilisim-hukuku-domain-dataset,1000,20.0,0.00,98.00,98.00
17,berkcangumusisik/voleykoc-identity-tr,182,26.0,0.00,85.71,85.71
20,enesozdemr/benim_ilk_datasetim,113,17.0,0.00,84.96,84.96
21,erhanalsr/langusta-identity,100,20.0,0.00,80.00,76.00
11,YusufSimsek/turkce-atasozleri-dataset,1398,446.0,0.21,68.10,67.95
35,samliumay/umay_samli_identification_dataset,219,73.0,0.00,66.67,66.67
40,sk75/sahin_identity,77,27.0,0.00,64.94,64.94
29,meldakahramann/animasyon-domain-dataset,1020,420.0,0.00,58.82,61.86
38,sedayzc/turkish-electronics-product-comparison...,11858,5662.0,41.37,52.25,55.96
13,aliFurkan123/identity,30,21.0,0.00,36.36,46.67


### Satır sayısı ile benzersiz içerik arasındaki fark

Satır sayısı bir kapasite göstergesidir, içerik hacmi değildir.
Aşağıdaki tablo aradaki farkın en büyük olduğu veri setlerini
sıralar; `distinct_answers` sütunu satır sayısının ne kadarının
gerçekten farklı cevaba karşılık geldiğini gösterir.

In [6]:
gap = summary.loc[summary["distinct_answers"].notna()].copy()
gap["answer_gap"] = gap["rows"] - gap["distinct_answers"]
gap.loc[gap["answer_gap"] > 0,
        ["dataset", "rows", "distinct_rows", "distinct_answers", "answer_gap"]
       ].sort_values("answer_gap", ascending=False).head(12)

,dataset,rows,distinct_rows,distinct_answers,answer_gap
38,sedayzc/turkish-electronics-product-comparison...,11858,7007,5662.0,6196.0
19,enes1863/bilisim-hukuku-domain-dataset,1000,1000,20.0,980.0
11,YusufSimsek/turkce-atasozleri-dataset,1398,1398,446.0,952.0
29,meldakahramann/animasyon-domain-dataset,1020,1020,420.0,600.0
31,nursimakgul/meb-soru-uretme,20874,20696,20609.0,265.0
32,nyzmemre/biyoloji-terimleri-turkce-sa,1093,1093,857.0,236.0
6,SalihHub/trendyol-marangoz-urun-asistan-qa,1211,1211,1028.0,183.0
17,berkcangumusisik/voleykoc-identity-tr,182,182,26.0,156.0
35,samliumay/umay_samli_identification_dataset,219,219,73.0,146.0
20,enesozdemr/benim_ilk_datasetim,113,113,17.0,96.0


### Veri setleri ne hakkında

Terimler koleksiyonun geri kalanına göre ağırlıklandırılmıştır, yani
sık geçeni değil ayırt edeni gösterir. Yoğunlaşma, en sık yirmi kökün
tüm konu token'ları içindeki payıdır: yüksek değer dar bir söz varlığı
demektir.

In [7]:
summary[["dataset", "rows", "topic_concentration_%", "distinctive_terms"]].sort_values(
    "topic_concentration_%", ascending=False
).head(15)

,dataset,rows,topic_concentration_%,distinctive_terms
1,Aysenur44/namaz-vakti-identity-tr,4,100.0,"yararsın, adın, kimsin, seni, geliştirdi"
20,enesozdemr/benim_ilk_datasetim,113,96.3,"kokulandırılır, petrol, boru, hattı, doğal"
2,Egertekin/marvel-domain-dataset,177,80.6,"örümcek, spider, adam, detaylı, marvel"
13,aliFurkan123/identity,30,75.5,"furkan, trained, tuned, goal, fine"
31,nursimakgul/meb-soru-uretme,20874,73.5,"üret, çoktan, sınıf, seçmeli, orta"
28,haticenurcakr/turkish-classic-books-qa,220,62.8,"eserinin, yayımlanmıştır, edebiyatında, yazarı..."
0,Aysenur44/namaz-vakti-dua-asistan-tr,60,58.9,"namazı, hocam, detaylı, kılınabilir, vakti"
21,erhanalsr/langusta-identity,100,56.3,"yanıtla, cümleyle, açıkça, soruyu, cevapla"
10,YusufSimsek/llm-kisisellestirme,46,53.5,"yusuf, birisi, ilgileniyor, senden, projeleri"
12,aliFurkan123/cultural-questions-dataset,500,52.3,"katının, başkenti, soracağım, kafadan, arkadaşlar"


### Aynı alanı kapsayan veri setleri

Söz varlığı benzerliği anlam benzerliği değildir; iki İngilizce veri
seti yalnız dil ortaklığı yüzünden yüksek skor alabilir. Her çift
yorumlanmadan önce okunmalıdır.

In [8]:
topics = json.loads(
    (PROJECT_DIR / "outputs" / "topic_overlap.json").read_text(encoding="utf-8")
)["pairs"]
pd.DataFrame(topics).head(12)

,left,right,cosine,shared_terms
0,seali/turkce-saglik-qa,senemde/saglik-qa-tr,0.5680,"[diyabetli, hastalığı, obezite, öneriler, teda..."
1,Erenyanic/seasoned-advice-dataset,gmz1234/stackoverflow_ai,0.3013,"[some, different, like, more, then, other]"
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,sedayzc/trendyol-electronics-products-features...,0.2700,"[kargo, satıcı, ürün, geldi, sipariş, aldım]"
3,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,SalihHub/trendyol-marangoz-urun-asistan-qa,0.2241,"[merhaba, kargo, acaba, ürün, göndermiş, istiy..."
4,Aysenur44/namaz-vakti-identity-tr,ssilistre/semih-silistre-ai-identity,0.2237,"[seni, adın, kimsin, geliştirdi]"
5,Aysenur44/namaz-vakti-identity-tr,berkcangumusisik/voleykoc-identity-tr,0.2226,"[yararsın, adın, kimsin, seni, geliştirdi]"
6,enes1863/bilisim-hukuku-domain-dataset,samliumay/turkish_cyber_security_controls_dataset,0.2103,"[veri, kişisel, bulut, gizlilik, hizmeti, kuru..."
7,berkbirkan/turkish-x-engagement-quotes,berkbirkan/turkish-x-engagement-replies,0.2055,"[aşağıdaki, gönderisini, kısa, alıntılarken, y..."
8,filiz-yalcin/identity-finetune,gmz1234/stackoverflow_ai,0.1998,"[function, some, other, model, more, different]"
9,SalihHub/trendyol-marangoz-urun-asistan-qa,namruni/meb-ogretmen-soru-cevap,0.1891,"[merhaba, acaba, istiyorum, teşekkürler, yapab..."


### Dil sinyali

Türkçeye özgü harflerin tüm harflere oranı ve durak sözcük sayımları
birlikte kaba bir dil sinyali üretir. Bu bir sınıflandırıcı değildir;
yalnız kapsam dışı kalan veri setlerini görünür kılar.

In [9]:
summary.groupby("language_signal")["dataset"].agg(["count", list])

,count,list
language_signal,,
english,2,"[aliFurkan123/identity, gmz1234/stackoverflow_ai]"
turkish,43,"[Aysenur44/namaz-vakti-dua-asistan-tr, Aysenur..."


### Yanıt uzunluğu görev biçimini etkiliyor

Medyan ve p95 kelime sayıları bağlam bütçesi, örnek ağırlığı ve hedef
yanıt biçimi için kullanılır; tek başına içerik doğruluğu göstermez.

In [10]:
summary.loc[
    summary["answer_words_median"].notna(),
    ["dataset", "answer_words_median", "answer_words_p95"],
].sort_values("answer_words_p95", ascending=False)

,dataset,answer_words_median,answer_words_p95
25,gmz1234/stackoverflow_ai,237.0,763.20
23,filiz-yalcin/identity-finetune,245.0,552.10
4,Erenyanic/seasoned-advice-dataset,121.0,366.10
44,yoitsmeyusuf/felsefe_finetune,35.0,353.60
36,seali/turkce-saglik-qa,151.0,330.40
38,sedayzc/turkish-electronics-product-comparison...,108.0,328.00
22,erhanalsr/langusta-kpss-reasoning,200.0,310.00
8,Toivo0/Turkce-istatistik-reasoning,219.5,278.00
33,sadecebirisii/turkish-llm-authority-bypass-saf...,218.0,270.60
16,berkcangumusisik/voleykoc-antrenorluk-tr,63.0,239.00


### Şema ve hazırlık sinyalleri ayrı izleniyor

Birebir tekrar, `thinking`, metin biçimindeki `null` ve zaman hassas
eşleşmeleri farklı düzeltme işlemleri gerektirir.

In [11]:
summary[[
    "dataset", "shape", "exact_duplicates", "thinking",
    "string_nulls", "time_sensitive_matches",
]].sort_values(
    ["string_nulls", "thinking", "time_sensitive_matches"],
    ascending=False,
)

,dataset,shape,exact_duplicates,thinking,string_nulls,time_sensitive_matches
40,sk75/sahin_identity,conversation,0,0,462,0
9,Uunan/turkish-cuisine-qa,conversation,0,34244,0,0
31,nursimakgul/meb-soru-uretme,conversation,178,20874,0,560
23,filiz-yalcin/identity-finetune,conversation,0,1600,0,1038
11,YusufSimsek/turkce-atasozleri-dataset,conversation,0,1398,0,0
6,SalihHub/trendyol-marangoz-urun-asistan-qa,conversation,0,1211,0,10
29,meldakahramann/animasyon-domain-dataset,conversation,0,1020,0,0
42,ssilistre/carnegie-dost-kazanma-tr,conversation,0,1001,0,154
4,Erenyanic/seasoned-advice-dataset,conversation,0,1000,0,28
19,enes1863/bilisim-hukuku-domain-dataset,conversation,0,1000,0,0


### Yapılandırılmış çıktı sözleşmesi ve çelişen hedefler

Bazı veri setleri cevabı JSON olarak tanımlar. `json_like_broken`,
JSON gibi başlayıp ayrıştırılamayan cevapları sayar; sözleşmenin
fiilen ne kadar tutulduğunu gösterir. `conflicting_prompt_families`
ise aynı normalleştirilmiş istemin birden fazla farklı cevapla
eşleştiği aile sayısıdır ve basit tekrardan ayrı bir sorundur.

In [12]:
summary.loc[
    (summary["json_answers"] > 0)
    | (summary["json_like_broken"] > 0)
    | (summary["conflicting_prompt_families"] > 0),
    ["dataset", "rows", "json_answers", "json_like_broken", "conflicting_prompt_families"],
].sort_values("json_like_broken", ascending=False)

,dataset,rows,json_answers,json_like_broken,conflicting_prompt_families
4,Erenyanic/seasoned-advice-dataset,1000,0,24,0
25,gmz1234/stackoverflow_ai,1000,0,2,0
39,senemde/saglik-qa-tr,553,0,1,0
2,Egertekin/marvel-domain-dataset,177,0,0,7
3,Endezyar/siyer_datasets,509,0,0,3
6,SalihHub/trendyol-marangoz-urun-asistan-qa,1211,0,0,12
14,berkbirkan/turkish-x-engagement-quotes,1000,0,0,229
15,berkbirkan/turkish-x-engagement-replies,1000,0,0,212
9,Uunan/turkish-cuisine-qa,34244,0,0,2
11,YusufSimsek/turkce-atasozleri-dataset,1398,0,0,3


### Veri setleri arası ortak metin aileleri

Ortak istem veya cevaplar birlikte veri hazırlarken aynı şablon
ailesi altında incelenmelidir.

In [13]:
pd.DataFrame(overlaps)

,left,right,shared_user_prompts,shared_assistant_answers,prompt_examples,answer_examples
0,Aysenur44/namaz-vakti-identity-tr,berkcangumusisik/voleykoc-identity-tr,1,0,[ne işe yararsın],[]
1,Aysenur44/namaz-vakti-identity-tr,gorkemergune/ayarlicazhocam_finetune,2,0,"[sen kimsin, seni kim geliştirdi]",[]
2,Aysenur44/namaz-vakti-identity-tr,samliumay/umay_samli_identification_dataset,3,0,"[adın ne, sen kimsin, seni kim geliştirdi]",[]
3,Aysenur44/namaz-vakti-identity-tr,sk75/sahin_identity,4,0,"[adın ne, ne işe yararsın, sen kimsin]",[]
4,Aysenur44/namaz-vakti-identity-tr,ssilistre/semih-silistre-ai-identity,3,0,"[adın ne, sen kimsin, seni kim geliştirdi]",[]
5,YusufSimsek/llm-kisisellestirme,erhanalsr/langusta-identity,1,0,[yaratıcın kim],[]
6,YusufSimsek/llm-kisisellestirme,samliumay/umay_samli_identification_dataset,1,0,[yaratıcın kim],[]
7,YusufSimsek/llm-kisisellestirme,sk75/sahin_identity,1,0,[yaratıcın kim],[]
8,YusufSimsek/llm-kisisellestirme,ssilistre/semih-silistre-ai-identity,1,0,[yaratıcın kim],[]
9,aliFurkan123/identity,berkcangumusisik/voleykoc-identity-tr,4,0,"[hi who are you, what is your name, what is yo...",[]


### İncelenmiş nitel bulgular

Bu tablo yalnız `manual_findings.json` içinde insan tarafından
gözden geçirilmiş bulguları gösterir. Yeni veri setleri bu dosyaya
eklenmeden burada otomatik olarak yorumlanmaz.

In [14]:
finding_rows = []
for dataset_id, detail in manual.get("datasets", {}).items():
    for finding in detail.get("findings", []):
        finding_rows.append({
            "dataset": dataset_id,
            "severity": finding.get("severity"),
            "evidence": finding.get("evidence"),
            "risk": finding.get("risk"),
            "remediation": finding.get("remediation"),
        })
findings = pd.DataFrame(finding_rows)
findings

,dataset,severity,evidence,risk,remediation
0,Uunan/turkish-cuisine-qa,high,Kart 2.714 benzersiz yemekten 34.244 soru-ceva...,Bir yemeğe ait hatalı bir olgu onlarca kayıtta...,"Yemek kimliğine göre kümele, örneklemi dengele..."
1,Uunan/turkish-cuisine-qa,medium,Asistan cevaplarının medyan uzunluğu 10.0 keli...,Thinking alanı doğrudan eğitilirse model gerek...,Yalnız nihai cevabı içeren türev sürüm üret; t...
2,nursimakgul/meb-soru-uretme,high,Dataset Viewer bu depoyu sunamıyor: meb_identi...,Veri seti standart yükleyicilerle açılamıyor; ...,Her satırı messages anahtarlı bir nesneye sar;...
3,nursimakgul/meb-soru-uretme,high,Cevapların 17.454/20.874 tanesi JSON; kalan 3....,Model aynı istem ailesi için bazen JSON bazen ...,Şemayı zorunlu kıl; JSON olmayan kayıtları ayr...
4,nursimakgul/meb-soru-uretme,high,"Normalleştirilmiş istem tekrarı %86,6 (18.084/...",Aynı sınıf/ders/konu isteminin farklı sorular ...,"İsteme zorluk, kazanım kodu ve soru tipi gibi ..."
...,...,...,...,...,...
82,Aysenur44/namaz-vakti-identity-tr,high,"Veri kartı gövdesi yok; kaynak, amaç ve sınırl...",Hedeflenen davranış ve veri kökeni belirsiz.,Tam bir veri kartı ekle.
83,Aysenur44/namaz-vakti-identity-tr,medium,Dört kullanıcı isteminin tamamı diğer kimlik v...,Kimlik setleri birleştirilirse aynı soru çeliş...,Kimlik setlerini birleştirme; tek personayı seç.
84,sedayzc/trendyol-electronics-products-features...,high,"Veri kartı yok. Kazıma tarihi, kapsam ölçütü v...","Fiyat, puan ve yorum sayısı kazıma anına aitti...","Kazıma tarihi, kapsam ve lisans bilgisini içer..."
85,sedayzc/trendyol-electronics-products-features...,medium,Eksik değerler alanlara dağılmış: comment_coun...,Eksik alanlar dönüşüm şablonlarında boş metin ...,Eksik değer politikası tanımla; zorunlu alanla...


## Çıkarımlar

- Her veri seti kullanım amacı, şeması ve kaynak izlenebilirliğiyle
  birlikte değerlendirilmelidir.
- Birebir tekrarın düşük olması, normalleştirilmiş istem veya cevap
  ailelerinin dengeli olduğu anlamına gelmez.
- Zaman hassas içerikler güncel ve doğrulanabilir kaynak katmanına
  bağlanmalıdır.
- Sohbet, düz istem–cevap, katalog ve genel tablo verileri aynı hedef
  şemaya zorlanmadan önce ayrı hazırlık yollarından geçirilmelidir.
- Nitel sonuçlar, satır örnekleri ve kaynak kartı incelenmeden yalnız
  otomatik profil değerlerinden türetilmemelidir.